# Model Training & Evaluation

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_ratings.csv")
print(f"Loaded: {len(df):,} ratings | "
      f"{df['book_id'].nunique():,} books | "
      f"{df['user_id'].nunique():,} users")

Loaded: 4,044,839 ratings | 22,931 books | 61,078 users


## 1. Train/Validation/Test Split

In [2]:
# Standard 80/10/10 train/val/test split
n = len(df)
test_size = int(n * 0.10)
val_size  = int(n * 0.10)

test  = df.sample(n=test_size, random_state=42)
rest  = df.drop(test.index)
val   = rest.sample(n=val_size, random_state=42)
train = rest.drop(val.index)

train = train.reset_index(drop=True)
val   = val.reset_index(drop=True)
test  = test.reset_index(drop=True)

print(f"Train: {len(train):,}  ({len(train)/n*100:.1f}%)")
print(f"Val:   {len(val):,}   ({len(val)/n*100:.1f}%)")
print(f"Test:  {len(test):,}   ({len(test)/n*100:.1f}%)")

# Cold-start check
for name, split in [("Val", val), ("Test", test)]:
    cold_u = set(split["user_id"]) - set(train["user_id"])
    cold_b = set(split["book_id"]) - set(train["book_id"])
    print(f"{name} cold-start — users: {len(cold_u)}, books: {len(cold_b)}")

train.to_csv("train_ratings.csv", index=False)
val.to_csv("val_ratings.csv",     index=False)
test.to_csv("test_ratings.csv",   index=False)
print("\nSaved: train_ratings.csv, val_ratings.csv, test_ratings.csv")

Train: 3,235,873  (80.0%)
Val:   404,483   (10.0%)
Test:  404,483   (10.0%)
Val cold-start — users: 0, books: 0
Test cold-start — users: 0, books: 0

Saved: train_ratings.csv, val_ratings.csv, test_ratings.csv


## 2. Train/Val/Test Validation

In [3]:
train = pd.read_csv("train_ratings.csv")
val   = pd.read_csv("val_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

ORIGINAL_TOTAL = 4_044_839
n = len(train) + len(val) + len(test)

print(f"{'CHECK':<45} RESULT")
print("─" * 68)

# 1. Total count preserved
status = "✓" if n == ORIGINAL_TOTAL else f"⚠ expected {ORIGINAL_TOTAL:,}"
print(f"{'1. Total count':<45} {n:,}  {status}")

# 2. Proportions (~80 / 10 / 10)
print(f"{'2. Proportions (train/val/test)':<45} "
      f"{len(train)/n*100:.1f}% / {len(val)/n*100:.1f}% / {len(test)/n*100:.1f}%")

# 3. No nulls
for name, d in [("Train", train), ("Val", val), ("Test", test)]:
    nulls = d.isnull().sum().sum()
    print(f"{'3. Nulls (' + name + ')':<45} {'✓ None' if nulls == 0 else f'⚠ {nulls}'}")

# 4. Rating range [1–5]
for name, d in [("Train", train), ("Val", val), ("Test", test)]:
    rmin, rmax = d["rating"].min(), d["rating"].max()
    ok = rmin >= 1 and rmax <= 5
    print(f"{'4. Rating range (' + name + ')':<45} {'✓' if ok else '⚠'} [{rmin}, {rmax}]")

# 5. No cross-split overlap
# Index-based splitting + total count preservation guarantees disjoint splits
# (same (user_id, book_id) pair cannot appear in two splits since original has no duplicates)
print(f"{'5. No cross-split overlap':<45} "
      f"{'✓ guaranteed (index-based split + total preserved)' if n == ORIGINAL_TOTAL else '⚠'}")

# 6. Cold-start check
cold_val_u  = len(set(val["user_id"])  - set(train["user_id"]))
cold_test_u = len(set(test["user_id"]) - set(train["user_id"]))
cold_val_b  = len(set(val["book_id"])  - set(train["book_id"]))
cold_test_b = len(set(test["book_id"]) - set(train["book_id"]))
print(f"{'6a. Cold-start users (val / test)':<45} {cold_val_u} / {cold_test_u}")
print(f"{'6b. Cold-start books (val / test)':<45} {cold_val_b} / {cold_test_b}")

# 7. Rating distribution (should be similar across splits)
print(f"\n{'7. Mean rating':<45} "
      f"Train: {train['rating'].mean():.3f} | "
      f"Val: {val['rating'].mean():.3f} | "
      f"Test: {test['rating'].mean():.3f}")
print(f"{'   Std rating':<45} "
      f"Train: {train['rating'].std():.3f}  | "
      f"Val: {val['rating'].std():.3f}  | "
      f"Test: {test['rating'].std():.3f}")

CHECK                                         RESULT
────────────────────────────────────────────────────────────────────
1. Total count                                4,044,839  ✓
2. Proportions (train/val/test)               80.0% / 10.0% / 10.0%
3. Nulls (Train)                              ✓ None
3. Nulls (Val)                                ✓ None
3. Nulls (Test)                               ✓ None
4. Rating range (Train)                       ✓ [1, 5]
4. Rating range (Val)                         ✓ [1, 5]
4. Rating range (Test)                        ✓ [1, 5]
5. No cross-split overlap                     ✓ guaranteed (index-based split + total preserved)
6a. Cold-start users (val / test)             0 / 0
6b. Cold-start books (val / test)             0 / 0

7. Mean rating                                Train: 3.988 | Val: 3.986 | Test: 3.987
   Std rating                                 Train: 0.937  | Val: 0.937  | Test: 0.939


## 3. Surprise SVD — Baseline

Train a default-parameter SVD on the full training set as a reference point before tuning.
SVD (matrix factorization) represents each user and book as a latent vector of length
`n_factors`; the predicted rating is their dot product plus global/user/book bias terms,
which absorb the dataset's strong positive skew (mean rating ≈ 3.99).

In [4]:
from surprise import SVD, Dataset, Reader, accuracy

reader = Reader(rating_scale=(1, 5))
train = pd.read_csv("train_ratings.csv")
test  = pd.read_csv("test_ratings.csv")

# Surprise trainset (full train) + test set as (user, book, rating) tuples
trainset = Dataset.load_from_df(train[["user_id", "book_id", "rating"]], reader).build_full_trainset()
testset  = list(test[["user_id", "book_id", "rating"]].itertuples(index=False, name=None))

baseline = SVD(random_state=42)  # defaults: n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02
baseline.fit(trainset)

rmse_baseline = accuracy.rmse(baseline.test(testset), verbose=False)
print(f"Baseline SVD (defaults) — test RMSE: {rmse_baseline:.4f}")

Baseline SVD (defaults) — test RMSE: 0.7614


## 4. Hyperparameter Tuning (GridSearchCV)

`GridSearchCV` runs 3-fold cross-validation over the parameter grid on the full training
set and keeps the combination with the lowest CV RMSE. `n_factors` is the number of
latent dimensions used to describe each user/book; `n_epochs`, `lr_all`, and `reg_all`
control training length, learning rate, and regularization strength.

In [5]:
import json
from surprise.model_selection import GridSearchCV

full_ds = Dataset.load_from_df(train[["user_id", "book_id", "rating"]], reader)

# Grid centered on the defaults (lr_all=0.005, reg_all=0.02) and extended outward
# (n_factors up to 200) so the optimum is bracketed inside the grid, not on a boundary.
param_grid = {
    "n_factors": [100, 150, 200],
    "n_epochs":  [20, 30, 40],
    "lr_all":    [0.005, 0.01, 0.02],
    "reg_all":   [0.02, 0.05, 0.1],
}

gs = GridSearchCV(SVD, param_grid, measures=["rmse"], cv=3, n_jobs=-1)
gs.fit(full_ds)  # 81 x 3 = 243 fits on the full 3.2M-row train set (~30 min)

best_params = gs.best_params["rmse"]
print(f"Best CV RMSE: {gs.best_score['rmse']:.4f}")
print(f"Best params:  {best_params}")
json.dump(best_params, open("best_params.json", "w"))

Best CV RMSE: 0.7574
Best params:  {'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.02, 'reg_all': 0.1}


## 5. Final Model & Evaluation

Refit SVD on the full training set with the tuned parameters and report RMSE on the
held-out test set, next to the baseline. On this large, sparse dataset most of the
predictable variance is already captured by the bias terms, so tuning moves test RMSE
only marginally — the two models are effectively tied. The fitted model is saved
(`svd_model.pkl`) for the console recommender.

In [6]:
import pickle

best_params = json.load(open("best_params.json"))
tuned = SVD(**best_params, random_state=42)
tuned.fit(trainset)
rmse_tuned = accuracy.rmse(tuned.test(testset), verbose=False)

print(f"Baseline test RMSE: {rmse_baseline:.4f}")
print(f"Tuned    test RMSE: {rmse_tuned:.4f}   (params: {best_params})")

# Save the tuned model for the console recommender (recommend.py)
with open("svd_model.pkl", "wb") as f:
    pickle.dump(tuned, f)
print("Saved final model -> svd_model.pkl")

Baseline test RMSE: 0.7614
Tuned    test RMSE: 0.7627   (params: {'n_factors': 200, 'n_epochs': 40, 'lr_all': 0.02, 'reg_all': 0.1})
Saved final model -> svd_model.pkl


## 6. Book Metadata Mapping

The ratings table only stores `book_id`. In a single pass over the Goodreads metadata
file (restricted to the books that survived cleaning) build two lookups for the console
recommender: `book_id → title` (to show real titles) and `book_id → work_id` (to collapse
different editions of the same work into one recommendation).

In [ ]:
import gzip, json

# Restrict the lookups to the books that survived cleaning (one pass over metadata)
cleaned_books = set(pd.read_csv("cleaned_ratings.csv", usecols=["book_id"])["book_id"].astype(str))

titles, works = {}, {}
with gzip.open("../goodreads_books_children.json.gz", "rt") as f:

    for line in f:
        rec = json.loads(line)
        bid = rec.get("book_id")
        if bid in cleaned_books:
            titles[bid] = (rec.get("title_without_series") or rec.get("title") or "").strip()
            works[bid] = rec.get("work_id") or bid   # editions of one work share a work_id

pd.DataFrame(sorted(titles.items()), columns=["book_id", "title"]).to_csv("book_titles.csv", index=False)
pd.DataFrame(sorted(works.items()), columns=["book_id", "work_id"]).to_csv("book_works.csv", index=False)

print(f"Saved {len(titles):,} titles -> book_titles.csv "
      f"({len(titles) / len(cleaned_books) * 100:.1f}% of {len(cleaned_books):,} cleaned books covered)")
print(f"Saved {len(works):,} book->work rows -> book_works.csv "
      f"({len(set(works.values())):,} distinct works)")

Saved 22,931 titles -> book_titles.csv (100.0% of 22,931 cleaned books covered)
Saved 22,931 book->work rows -> book_works.csv (20,471 distinct works)


## 7. Cross-Model Ranking Evaluation (graded NDCG@10)

RMSE measures rating accuracy, but the TwoTower model is ranking-only and cannot produce
RMSE, so the fair cross-model comparison uses **graded NDCG@10** on shared candidate pools:
each user's held-out test books (graded by their true rating) plus 100 sampled negatives.
The pool builder and metric are the *same functions* used in `twotower.ipynb` (seed=42), so
the numbers are directly comparable. SVD scores each candidate by its predicted rating (`.est`).

The tuned SVD ranks poorly (NDCG ≈ 0.16) despite its good RMSE (0.76) — a known RMSE–NDCG
divergence. SVD's predictions cluster near the global mean (~3.8), so positives and negatives
are hard to separate when the predicted rating is used as a ranking score.

In [10]:
from sklearn.metrics import ndcg_score

RANDOM_STATE = 42

# Shared eval structures — identical to twotower.ipynb so the two models compare fairly
train_items_by_user = train.groupby("user_id")["book_id"].apply(set).to_dict()
test_ratings_by_user = (
    test.groupby("user_id")
        .apply(lambda g: dict(zip(g["book_id"], g["rating"])), include_groups=False)
        .to_dict()
)
all_items = train["book_id"].unique()


def build_candidate_pools(test_ratings_by_user, train_items_by_user, all_items,
                          n_neg=100, seed=RANDOM_STATE):
    """Each user's pool = held-out test items (graded positives) + n_neg sampled negatives
    (non-interacted). Built once with a fixed seed and shared across models."""
    rng = np.random.default_rng(seed)
    items_arr = np.asarray(all_items)
    pools = {}
    for user, test_items in test_ratings_by_user.items():
        exclude = train_items_by_user.get(user, set()) | set(test_items.keys())
        negs = []
        while len(negs) < n_neg:
            cand = rng.choice(items_arr, size=n_neg * 2, replace=False)
            negs = [it for it in cand if it not in exclude][:n_neg]
        pools[user] = list(test_items.keys()) + negs
    return pools


def evaluate_ndcg(score_fn, pools, test_ratings_by_user, k=10):
    """Mean graded NDCG@k. Relevance = true test rating (0 for sampled negatives)."""
    ndcgs = []
    for user, items in pools.items():
        ratings = test_ratings_by_user[user]
        y_true = [ratings.get(it, 0) for it in items]
        if sum(y_true) == 0 or len(set(y_true)) == 1:   # need >=1 positive and some variation
            continue
        ndcgs.append(ndcg_score([y_true], [score_fn(user, items)], k=k))
    return float(np.mean(ndcgs)), len(ndcgs)


candidate_pools = build_candidate_pools(test_ratings_by_user, train_items_by_user, all_items)

# SVD ranks candidates by its predicted rating
svd_score_fn = lambda user, items: [tuned.predict(user, it).est for it in items]
ndcg, n_users = evaluate_ndcg(svd_score_fn, candidate_pools, test_ratings_by_user)

print(f"Tuned SVD graded NDCG@10: {ndcg:.4f}  (over {n_users:,} users)")
print(f"For reference — tuned SVD test RMSE: {rmse_tuned:.4f}")

Tuned SVD graded NDCG@10: 0.1628  (over 58,787 users)
For reference — tuned SVD test RMSE: 0.7627


## 8. Rating Accuracy — RMSE vs a Naive Baseline

RMSE is SVD's **native** rating-prediction metric (NOT a cross-model comparison — TwoTower has
no RMSE). To judge whether SVD's RMSE reflects real learning, we compare it to a **global-mean
baseline** (predict the train mean for every test rating), whose RMSE equals the rating std.

In [ ]:
# Global-mean baseline: predict the train mean for every test rating.
# Answers "does SVD learn signal beyond a no-information predictor?"
global_mean = train["rating"].mean()
baseline_rmse = np.sqrt(np.mean((test["rating"] - global_mean) ** 2))

# SVD's own RMSE on the test set (rating-prediction task; no candidate pools)
testset = list(test[["user_id", "book_id", "rating"]].itertuples(index=False, name=None))
svd_rmse = accuracy.rmse(tuned.test(testset), verbose=False)
svd_mae  = accuracy.mae(tuned.test(testset), verbose=False)

print(f"Global-mean baseline test RMSE: {baseline_rmse:.4f}")
print(f"SVD (default) test RMSE:        {svd_rmse:.4f}")
print(f"SVD (default) test MAE:         {svd_mae:.4f}")
print(f"Improvement over naive baseline: {(baseline_rmse - svd_rmse)/baseline_rmse*100:.1f}%")
print(f"(SVD: RMSE 0.7614 default / 0.7627 tuned)")

Global-mean baseline test RMSE: 0.9388
SVD (default) test RMSE:        0.7614
SVD (default) test MAE:         0.5848
Improvement over naive baseline: 18.9%
(Kate's SVD: RMSE 0.7614 default / 0.7627 tuned)


## 9. Graded NDCG — SVD as a Ranking Model (ranking-ization)

Re-score with the raw latent-vector dot product `pu·qi` (no biases/global mean) — the direct
analogue of TwoTower's similarity. Controls the "usage" variable to isolate **architecture**.

In [11]:
# === Tuned SVD as a RANKING model (pu·qi) — self-contained for svd.ipynb ===
# svd.ipynb only has the default `svd_model`. This cell trains a TUNED SVD with the
# same best params found in model.ipynb, then scores it as a ranking model (pu·qi).

# 1. Train tuned SVD with the GridSearchCV best params (from model.ipynb)
best_params = {"n_factors": 200, "n_epochs": 40, "lr_all": 0.02, "reg_all": 0.1}
tuned = SVD(**best_params, random_state=RANDOM_STATE)
tuned.fit(trainset)            # reuse the trainset already built in this notebook
print("Tuned SVD trained")

# 2. Score tuned SVD as a ranking model: raw pu·qi dot product (no biases)
def tuned_svd_ranking_score_fn(user, items):
    inner_u = trainset.to_inner_uid(user)
    u_vec = tuned.pu[inner_u]
    scores = []
    for it in items:
        try:
            inner_i = trainset.to_inner_iid(it)
            scores.append(float(np.dot(u_vec, tuned.qi[inner_i])))
        except ValueError:
            scores.append(-np.inf)
    return scores

tuned_rank_ndcg, n_users = evaluate_ndcg(
    tuned_svd_ranking_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"Tuned SVD (as ranking, pu.qi) — graded NDCG@10: {tuned_rank_ndcg:.4f}  (over {n_users:,} users)")

# 3. Report tuned SVD as a rating model, to confirm 0.1628
def tuned_svd_rating_score_fn(user, items):
    return [tuned.predict(user, it).est for it in items]

tuned_rating_ndcg, _ = evaluate_ndcg(
    tuned_svd_rating_score_fn, candidate_pools, test_ratings_by_user, k=10
)
print(f"Tuned SVD (as rating, est)   — graded NDCG@10: {tuned_rating_ndcg:.4f}  (should be ~0.1628)")

Tuned SVD trained
Tuned SVD (as ranking, pu.qi) — graded NDCG@10: 0.2242  (over 58,787 users)
Tuned SVD (as rating, est)   — graded NDCG@10: 0.1628  (should be ~0.1628)


## 10. Diagnostic — Why SVD Ranks Poorly (RMSE–NDCG divergence)

SVD has good RMSE but poor NDCG. These diagnostics show why: its predicted ratings cluster near
the global mean, so positives and negatives are barely separated.

In [12]:
# What does SVD predict for one user's pool? (sanity check)
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]
preds = [tuned.predict(u0, it) for it in items0[:10]]
for p in preds:
    print(f"item={p.iid}  est={p.est:.4f}  impossible={p.details.get('was_impossible')}")

item=6250208  est=3.5169  impossible=False
item=297676  est=3.4436  impossible=False
item=1258121  est=3.5787  impossible=False
item=3636  est=4.1429  impossible=False
item=11788115  est=3.5385  impossible=False
item=14823888  est=3.5414  impossible=False
item=982432  est=3.9119  impossible=False
item=20307024  est=4.0363  impossible=False
item=23600172  est=3.4760  impossible=False
item=92637  est=3.9380  impossible=False


In [13]:
# Are positives scored higher than negatives for one user? By how much?
u0 = next(iter(candidate_pools))
items0 = candidate_pools[u0]
ratings0 = test_ratings_by_user[u0]

pos_est, neg_est = [], []
for it in items0:
    est = tuned.predict(u0, it).est
    (pos_est if it in ratings0 else neg_est).append(est)

print(f"User {u0[:8]}")
print(f"  positives: {len(pos_est)} items, est mean={np.mean(pos_est):.3f}, range=[{min(pos_est):.2f}, {max(pos_est):.2f}]")
print(f"  negatives: {len(neg_est)} items, est mean={np.mean(neg_est):.3f}, range=[{min(neg_est):.2f}, {max(neg_est):.2f}]")
print(f"  -> positives scored higher? {np.mean(pos_est) > np.mean(neg_est)}")

User 00004584
  positives: 4 items, est mean=3.671, range=[3.44, 4.14]
  negatives: 100 items, est mean=3.745, range=[3.07, 4.52]
  -> positives scored higher? False


In [14]:
# Is the pos/neg gap small across MANY users (not just one)?
gaps = []
for user, items in list(candidate_pools.items())[:2000]:
    ratings = test_ratings_by_user[user]
    pos, neg = [], []
    for it in items:
        est = tuned.predict(user, it).est
        (pos if it in ratings else neg).append(est)
    if pos and neg:
        gaps.append(np.mean(pos) - np.mean(neg))

gaps = np.array(gaps)
print(f"Over {len(gaps)} users:")
print(f"  mean (pos_est - neg_est): {gaps.mean():.4f}")
print(f"  fraction where positives scored higher: {(gaps > 0).mean():.1%}")
print("Interpretation: right direction (>50%) but weak separation (~0.1 on a 1-5 scale)")
print("-> good RMSE, poor ranking (the RMSE-NDCG divergence).")

Over 2000 users:
  mean (pos_est - neg_est): 0.1447
  fraction where positives scored higher: 85.0%
Interpretation: right direction (>50%) but weak separation (~0.1 on a 1-5 scale)
-> good RMSE, poor ranking (the RMSE-NDCG divergence).
